In [ ]:
import io
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import requests

### UCI Electricity Load Dataset

In [ ]:
# Download data

UCI_DATA_URL = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"

def load_uci_data(
    target_dir: str,
    target_file: str | None = "LD2011_2014.txt",
    url: str = UCI_DATA_URL
):
    response = requests.get(UCI_DATA_URL)
    try: 
        response.raise_for_status()
    except requests.HTTPError as e:
        print(f"Failed to download from {url}. Reason: {e}")
        return 

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        try: 
            if target_file:
                z.extract(member=target_file, path=target_dir)
            else: 
                z.extractall(path=target_dir)
        except Exception as e:
            print(f"Failed to extract files. Reason: {e}")
            return


In [ ]:
# Load data as polars dataframe
df = pl.read_csv(
    "./data/LD2011_2014.txt",
    has_header=True,
    separator=";",
    decimal_comma=True,
    try_parse_dates=True,
    infer_schema_length=1_000_000
)

# First column should be timestamp column
df = df.rename({df.columns[0]: "timestamp"}).sort(by="timestamp")

### EDA

In [ ]:
min_timestamp = df.get_column("timestamp").min()
max_timestamp = df.get_column("timestamp").max()

expected_timestamps = pl.datetime_range(
    start=min_timestamp,
    end=max_timestamp,
    interval="15m",
    closed="both",
    eager=True,
)

print("Expected number of timestamps: ", len(expected_timestamps))
print("Actual number of timestamps: ", len(df))

In [ ]:
# Number of non-zero observations

# TODO: Add min / max counts!
non_zero_counts = (df.select(pl.exclude("timestamp")) > 0).sum()

plt.hist(non_zero_counts.to_numpy().flatten(), bins=20, color="tab:blue", alpha=0.75)
plt.xlabel("Non-zero count")
plt.ylabel("Frequency");

In [ ]:
# First and last observations for each client
first_last_timestamp_by_client = (
    df.unpivot(
        on=[c for c in df.columns if c != "timestamp"],
        index="timestamp",
        variable_name="client"
    )
    .filter(pl.col("value") > 0)
    .group_by("client", maintain_order=True)
    .agg(
        min_timestamp=pl.col("timestamp").min(),
        max_timestamp=pl.col("timestamp").max()
    )
)

# Plot
first_ts = first_last_timestamp_by_client["min_timestamp"].to_list()
last_ts = first_last_timestamp_by_client["max_timestamp"].to_list()

fig, ax = plt.subplots(1, 1)
ax.scatter(first_ts, np.arange(len(first_ts)), alpha=0.1, color="tab:blue", label="first")
ax.scatter(last_ts, np.arange(len(last_ts)), alpha=0.1, color="tab:red", label="last")
ax.legend()
for tick in ax.get_xticklabels():
    tick.set_rotation(45)
ax.set(xlabel="Timestamp", ylabel="Client")
fig.align_labels()
fig.tight_layout();

In [ ]:
# NEXT QUESTIONS:

# Are timestamps continuous?
# What do some of the timeseries actually look like?
# Timeseries clustering?